# 01 — Auditoría de datos
StreamView Analytics · ADY1104 · Matías Retamal · Claudio González

Objetivo: cargar los dos CSV crudos y verificar en código, celda por celda, los ocho hallazgos declarados en el README. Nada de este notebook modifica `data/raw/`.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW = '../data/raw/'
movies = pd.read_csv(RAW + 'netflix_movies_detailed_up_to_2025.csv')
tv = pd.read_csv(RAW + 'netflix_tv_shows_detailed_up_to_2025.csv')

print('movies:', movies.shape)
print('tv:', tv.shape)

movies: (16000, 18)
tv: (16000, 16)


## Shape, dtypes y head

In [2]:
movies.dtypes

show_id           int64
type                str
title               str
director            str
cast                str
country             str
date_added          str
release_year      int64
rating          float64
duration        float64
genres              str
language            str
description         str
popularity      float64
vote_count        int64
vote_average    float64
budget            int64
revenue           int64
dtype: object

In [3]:
movies.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000,752600867
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000,839030630
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000,954305868
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.600,NaN,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000,592461732
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.800,NaN,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000,494879471


In [4]:
tv.dtypes

show_id           int64
type                str
title               str
director            str
cast                str
country             str
date_added          str
release_year      int64
rating          float64
duration            str
genres              str
language            str
description         str
popularity      float64
vote_count        int64
vote_average    float64
dtype: object

In [5]:
tv.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average
0,33238,TV Show,Running Man,안재철,"Yoo Jae-suk, Jee Seok-jin, Kim Jong-kook, Haha...",South Korea,2010-07-11,2010,8.241,1 Seasons,"Comedy, Reality",ko,A reality and competition show where members a...,1929.898,187,8.241
1,32415,TV Show,Conan,NaN,"Conan O'Brien, Andy Richter",United States of America,2010-11-08,2010,7.035,1 Seasons,"Talk, Comedy, News",en,A late night television talk show hosted by C...,1670.580,229,7.035
2,37757,TV Show,MasterChef Greece,NaN,NaN,Greece,2010-10-03,2010,5.600,1 Seasons,Reality,el,MasterChef Greece is a Greek competitive cooki...,1317.092,6,5.600
3,75685,TV Show,Prostřeno!,NaN,"Václav Vydra, Jana Boušková",Czech Republic,2010-03-01,2010,6.500,1 Seasons,Reality,cs,The knives (and forks) are out as a group of s...,1095.776,6,6.500
4,33847,TV Show,The Talk,NaN,"Amanda Kloots, Jerry O'Connell, Akbar Gbaja-Bi...","United States of America, Ireland",2010-10-18,2010,3.400,1 Seasons,Talk,en,A panel of well-known news and entertainment p...,712.070,12,3.400


## % de nulos por columna
Salida que va pegada en `docs/calidad_datos.md`, sección 2.

In [6]:
nulos_movies = (movies.isna().mean() * 100).round(2)
nulos_tv = (tv.isna().mean() * 100).round(2)
print('Nulos MOVIES (%)')
print(nulos_movies)
print('\nNulos TV (%)')
print(nulos_tv)

Nulos MOVIES (%)
show_id           0.00
type              0.00
title             0.00
director          0.82
cast              1.27
country           2.91
date_added        0.00
release_year      0.00
rating            0.00
duration        100.00
genres            0.67
language          0.00
description       0.82
popularity        0.00
vote_count        0.00
vote_average      0.00
budget            0.00
revenue           0.00
dtype: float64

Nulos TV (%)
show_id          0.00
type             0.00
title            0.00
director        68.53
cast             7.23
country         11.23
date_added       0.00
release_year     0.00
rating           0.00
duration         0.00
genres           6.09
language         0.00
description     20.04
popularity       0.00
vote_count       0.00
vote_average     0.00
dtype: float64


## Hallazgo 1 — `rating` es copia exacta de `vote_average`

In [7]:
print('movies: rating == vote_average en', (movies['rating'] == movies['vote_average']).mean()*100, '% de filas')
print('tv: rating == vote_average en', (tv['rating'] == tv['vote_average']).mean()*100, '% de filas')

movies: rating == vote_average en 100.0 % de filas
tv: rating == vote_average en 100.0 % de filas


**Conclusión:** confirmado al 100% en ambos archivos. No existe clasificación etaria en el dataset — `rating` fue sobrescrita. Se elimina en la limpieza y se descarta el análisis de clasificación por edad.

## Hallazgo 2 — `duration` inservible

In [8]:
print('movies: % nulo en duration =', movies['duration'].isna().mean()*100)
print('tv: valores únicos en duration =', tv['duration'].unique())

movies: % nulo en duration = 100.0
tv: valores únicos en duration = <StringArray>
['1 Seasons']
Length: 1, dtype: str


**Conclusión:** `duration` es 100% nula en películas y constante ("1 Seasons") en las 16.000 series. Cero varianza en ambos casos: el análisis de duración vs popularidad pedido en la infografía es imposible con este dataset. Se descarta explícitamente y se elimina la columna.

## Hallazgo 3 — exactamente 1.000 títulos por año (2010–2025)

In [9]:
print('MOVIES por release_year:')
print(movies['release_year'].value_counts().sort_index())
print('\nTV por release_year:')
print(tv['release_year'].value_counts().sort_index())

MOVIES por release_year:
release_year
2010    1000
2011    1000
2012    1000
2013    1000
2014    1000
2015    1000
2016    1000
2017    1000
2018    1000
2019    1000
2020    1000
2021    1000
2022    1000
2023    1000
2024    1000
2025    1000
Name: count, dtype: int64

TV por release_year:
release_year
2010    1000
2011    1000
2012    1000
2013    1000
2014    1000
2015    1000
2016    1000
2017    1000
2018    1000
2019    1000
2020    1000
2021    1000
2022    1000
2023    1000
2024    1000
2025    1000
Name: count, dtype: int64


**Conclusión:** exactamente 1.000 títulos por año, en ambos archivos, sin excepción. Es un artefacto del muestreo del dataset, no un hallazgo de negocio. Por eso no se grafica evolución del volumen del catálogo en el tiempo — cualquier gráfico de conteo por año saldría plano por construcción, no por realidad de negocio.

## Hallazgo adicional — `release_year` == año de `date_added`

In [10]:
m_year = pd.to_datetime(movies['date_added'], errors='coerce').dt.year
t_year = pd.to_datetime(tv['date_added'], errors='coerce').dt.year
print('movies: coincide en', (m_year == movies['release_year']).mean()*100, '% de filas')
print('tv: coincide en', (t_year == tv['release_year']).mean()*100, '% de filas')

movies: coincide en 100.0 % de filas
tv: coincide en 100.0 % de filas


**Conclusión:** coincide en el 100% de los casos en ambos archivos. No se puede medir antigüedad del contenido al momento de incorporarse al catálogo — se descarta ese análisis.

## Hallazgo 4 — `budget`/`revenue` en cero (solo películas)

In [11]:
print('budget == 0:', (movies['budget']==0).mean()*100, '%')
print('revenue == 0:', (movies['revenue']==0).mean()*100, '%')
subset_roi = movies[(movies['budget']>0) & (movies['revenue']>0)]
print('Películas con ambos > 0 (subconjunto válido para ROI):', len(subset_roi))

budget == 0: 69.70625 %
revenue == 0: 64.71875 %
Películas con ambos > 0 (subconjunto válido para ROI): 3540


**Conclusión:** 69,7% de las películas tiene `budget`=0 y 64,7% tiene `revenue`=0. Solo 3.540 películas (22,1%) tienen ambos valores positivos y sirven para calcular ROI. El análisis financiero se limita a ese subconjunto y se declara explícitamente en el título de cualquier gráfico que lo use.

## Hallazgo 5 — duplicados de `show_id`

In [12]:
print('Duplicados dentro de MOVIES:', movies['show_id'].duplicated().sum())
print('Duplicados dentro de TV:', tv['show_id'].duplicated().sum())
colisiones = set(movies['show_id']) & set(tv['show_id'])
print('Colisiones cruzadas (mismo show_id en ambos archivos):', len(colisiones))

Duplicados dentro de MOVIES: 0
Duplicados dentro de TV: 9
Colisiones cruzadas (mismo show_id en ambos archivos): 397


**Conclusión:** 9 duplicados internos en TV y 397 colisiones entre archivos. Sin prefijar los IDs (`MOV_` / `TV_`), al concatenar ambos datasets se mezclarían 397 registros de películas y series bajo el mismo identificador.

## Hallazgo 6 — `vote_count` en cero y elección del umbral mínimo

In [13]:
print('vote_count == 0 — movies:', (movies['vote_count']==0).mean()*100, '%')
print('vote_count == 0 — tv:', (tv['vote_count']==0).mean()*100, '%')

print('\nPercentiles movies:')
print(movies['vote_count'].describe(percentiles=[.05,.1,.25,.5,.75,.9,.95]))
print('\nPercentiles tv:')
print(tv['vote_count'].describe(percentiles=[.05,.1,.25,.5,.75,.9,.95]))

vote_count == 0 — movies: 5.5875 %
vote_count == 0 — tv: 22.9625 %

Percentiles movies:
count    16000.000000
mean       718.656125
std       2080.198316
min          0.000000
5%           0.000000
10%          6.000000
25%         53.000000
50%        138.000000
75%        422.000000
90%       1519.000000
95%       3356.050000
max      37119.000000
Name: vote_count, dtype: float64

Percentiles tv:
count    16000.000000
mean       107.014313
std        607.461581
min          0.000000
5%           0.000000
10%          0.000000
25%          1.000000
50%          4.000000
75%         30.000000
90%        167.000000
95%        409.050000
max      24664.000000
Name: vote_count, dtype: float64


**Decisión y motivo:** el umbral existe para sacar de los rankings la cola de votación anecdótica — el 5,6% de las películas y el 23,0% de las series tienen `vote_count` = 0, lo que arrastra su `vote_average` a 0.0 y ensucia cualquier orden por nota.

El umbral es **por tipo** porque las dos distribuciones no son comparables: mediana de 138 votos en películas contra 4 en series. Un umbral único de 30 excluiría el 17,4% de las películas pero el 75,0% de las series, vaciando la muestra. Se fija `vote_count >= 30` para películas y `vote_count >= 5` para series: cada corte deja fuera los títulos sin votación real conservando una muestra utilizable de ambos tipos.

In [14]:
print('Movies con umbral >=30: quedan', (movies['vote_count']>=30).sum(), 'de', len(movies))
print('TV con umbral >=5: quedan', (tv['vote_count']>=5).sum(), 'de', len(tv))

Movies con umbral >=30: quedan 13217 de 16000
TV con umbral >=5: quedan 7763 de 16000


## Hallazgo 7 — nulos altos en `director`, `description`, `country` (series)

In [15]:
cols = ['director', 'description', 'country', 'cast', 'genres']
print('TV:')
print((tv[cols].isna().mean()*100).round(1))
print('\nMOVIES:')
print((movies[cols].isna().mean()*100).round(1))

TV:
director       68.5
description    20.0
country        11.2
cast            7.2
genres          6.1
dtype: float64

MOVIES:
director       0.8
description    0.8
country        2.9
cast           1.3
genres         0.7
dtype: float64


**Conclusión:** `director` tiene 68,5% de nulos en series (vs 0,8% en películas). El análisis por director se restringe a películas. `description` (20%) y `country` (11,2%) también limitan análisis de texto y geografía en series, pero no los invalidan.

---
## Resumen para `docs/calidad_datos.md`

Todos los ocho hallazgos del README quedan confirmados en código. Los tres críticos (duration, rating=vote_average, 1.000/año) están cuantificados arriba con exactitud. El umbral de `vote_count` queda definido y justificado: 30 para películas, 5 para series.